# RBS Strength vs. Evo2 Log-Likelihood — Kosuri Dataset

Correlation analysis between Evo2 negative log-likelihood (scored on `trimmed_sequence`:
last 20 bp of upstream context + RBS) and experimentally measured relative translation
rate (`mean_xlat`) from Kosuri et al. 2013.

Data: `data/processed/kosuri_contexts_labeled.csv`  
Scored sequences: manually labeled via Evo2 online interface.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, kendalltau

sns.set_theme(style="whitegrid", context="notebook")

DATA_PATH  = "../data/processed/kosuri_contexts_labeled.csv"
OUTPUT_DIR = "../results/rbs_kosuri_scoring"

## Load Data

In [ ]:
df_all = pd.read_csv(DATA_PATH)
df = df_all.dropna(subset=["evo2_neg_log_likelihood", "mean_xlat"]).copy()
df["log10_xlat"] = np.log10(df["mean_xlat"])
# Convert NLL to log-prob (negate) for intuitive direction: higher = more likely
df["evo2_log_prob"] = -df["evo2_neg_log_likelihood"]

print(f"Total rows in CSV:  {len(df_all)}")
print(f"Labeled (used):     {len(df)}")
print(f"Unlabeled (skipped): {len(df_all) - len(df)}")
print()
print(f"NLL range:       {df['evo2_neg_log_likelihood'].min():.4f} – {df['evo2_neg_log_likelihood'].max():.4f}")
print(f"log-prob range:  {df['evo2_log_prob'].min():.4f} – {df['evo2_log_prob'].max():.4f}")
print(f"mean_xlat range: {df['mean_xlat'].min():.0f} – {df['mean_xlat'].max():.0f}  ({df['mean_xlat'].max()/df['mean_xlat'].min():.0f}× span)")

df[["rbs_name", "rbs_sequence", "mean_xlat", "evo2_neg_log_likelihood"]].head(8)

## Statistical Tests

In [ ]:
x   = df["evo2_log_prob"].values   # higher = more likely under Evo2
y   = df["mean_xlat"].values
ly  = df["log10_xlat"].values
n   = len(df)
α   = 0.05

r,    p_r    = pearsonr(x, y)
r_l,  p_rl   = pearsonr(x, ly)
rho,  p_rho  = spearmanr(x, y)
tau,  p_tau  = kendalltau(x, y)

rows = [
    ("Pearson r",    "evo2_log_prob vs mean_xlat (raw)",         r,   p_r),
    ("Pearson r",    "evo2_log_prob vs log₁₀(mean_xlat)",        r_l, p_rl),
    ("Spearman ρ",   "evo2_log_prob vs mean_xlat",               rho, p_rho),
    ("Kendall τ",    "evo2_log_prob vs mean_xlat",               tau, p_tau),
]

stats = pd.DataFrame(rows, columns=["Test", "Variables", "Coefficient", "p-value"])
stats["Significant"] = stats["p-value"].map(lambda p: "Yes" if p < α else "No")

print(f"n = {n}  |  α = {α}")
print()
for _, row in stats.iterrows():
    sig = "*" if row["Significant"] == "Yes" else " "
    print(f"{sig} {row['Test']:12s}  {row['Coefficient']:+.4f}   p={row['p-value']:.4f}   {row['Variables']}")

## Scatter Plots

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    f"Evo2 Log-Prob vs. Translation Rate — Kosuri RBS Library\n"
    f"trimmed_sequence scoring (20 bp upstream + RBS) | n={n}",
    fontsize=12,
)

for ax, (yvals, ylabel, coeff, pval, color) in zip(axes, [
    (y,  "mean_xlat",          r,   p_r,  "steelblue"),
    (ly, "log₁₀(mean_xlat)",  r_l, p_rl, "mediumseagreen"),
]):
    ax.scatter(x, yvals, alpha=0.65, edgecolors="none", s=45, color=color)

    m, b = np.polyfit(x, yvals, 1)
    xline = np.linspace(x.min(), x.max(), 200)
    ax.plot(xline, m * xline + b, color="tomato", linewidth=1.5)

    # Label extreme points
    top2 = df.nlargest(2, "mean_xlat")
    bot2 = df.nsmallest(2, "mean_xlat")
    for _, row in pd.concat([top2, bot2]).iterrows():
        yval = np.log10(row["mean_xlat"]) if ylabel != "mean_xlat" else row["mean_xlat"]
        ax.annotate(
            row["rbs_name"],
            (row["evo2_log_prob"], yval),
            fontsize=7, xytext=(5, 3), textcoords="offset points",
        )

    ax.set_xlabel("Evo2 log-prob (mean per token, negated NLL)")
    ax.set_ylabel(ylabel)
    ax.set_title(f"r = {coeff:+.3f}   p = {pval:.4f}{'  *' if pval < α else ''}")

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "rbs_kosuri_correlation.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"Saved: {fig_path}")
plt.show()

## NLL Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["evo2_neg_log_likelihood"], bins=20, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Evo2 NLL (per token)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"NLL distribution  (range: {df['evo2_neg_log_likelihood'].min():.3f}–{df['evo2_neg_log_likelihood'].max():.3f})")

axes[1].hist(np.log10(df["mean_xlat"]), bins=20, color="mediumseagreen", edgecolor="white")
axes[1].set_xlabel("log₁₀(mean_xlat)")
axes[1].set_ylabel("Count")
axes[1].set_title("Translation rate distribution")

plt.tight_layout()
plt.show()

## Save Stats

In [ ]:
stats_path = os.path.join(OUTPUT_DIR, "rbs_kosuri_stats.csv")
stats.to_csv(stats_path, index=False)
print(f"Saved: {stats_path}")